# Modeling and Evaluation

This notebook consumes the window-level feature table produced by Person 2. The expected input is `data/processed/model_features.csv` with a `condition` column, numeric EEG features, and optional `subject_id`, `session_id`, and `window_id` metadata.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == '03_modeling':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_feature_data
from src.personal_baseline.build_baseline import build_personal_baseline
from src.models.general_model import train_general_models, save_general_model
from src.models.personalized_model import train_personalized_models, save_personalized_model
from src.evaluation.evaluate_models import (
    calculate_metrics,
    compare_models,
    save_comparison_report,
    create_confusion_matrix,
)

In [ ]:
feature_path = PROJECT_ROOT / 'data' / 'processed' / 'model_features.csv'
data = load_feature_data(feature_path)
print(f'Loaded {len(data):,} windows and {len(data.columns):,} columns')
display(data.head())

## General population model

When `subject_id` is available, the trainer evaluates on held-out subjects to avoid leaking subject-specific patterns across the split.

In [ ]:
(general_model, general_name, general_features, general_X_test, general_y_test, general_results) = train_general_models(data)
save_general_model(
    general_model,
    general_name,
    general_features,
    str(PROJECT_ROOT / 'models' / 'general_model.pkl')
)

## Personalized model

The personalized model is trained for one subject using that subject's `rest` windows as the baseline. Set `subject_id` to the subject whose baseline should be used.

In [ ]:
if 'subject_id' not in data.columns:
    raise ValueError('Person 2 output must include subject_id for personalized modeling.')

subject_id = data['subject_id'].iloc[0]
subject_data = data[data['subject_id'].astype(str) == str(subject_id)].copy()
baseline = build_personal_baseline(subject_data, subject_id)
(personalized_model, personalized_name, personalized_features, personalized_X_test, personalized_y_test, personalized_results) = train_personalized_models(subject_data, baseline)
save_personalized_model(
    personalized_model,
    personalized_name,
    personalized_features,
    baseline,
    str(PROJECT_ROOT / 'models' / 'personalized_model.pkl')
)

In [ ]:
general_metrics = calculate_metrics(
    general_y_test,
    general_model.predict(general_X_test)
)
personalized_metrics = calculate_metrics(
    personalized_y_test,
    personalized_model.predict(personalized_X_test)
)
comparison = compare_models(general_metrics, personalized_metrics)
display(comparison)
save_comparison_report(
    comparison,
    str(PROJECT_ROOT / 'results' / 'reports' / 'model_comparison.csv')
)
create_confusion_matrix(
    general_y_test,
    general_model.predict(general_X_test),
    title='General Model Confusion Matrix',
    output_path=str(PROJECT_ROOT / 'results' / 'figures' / 'general_confusion_matrix.png')
)
create_confusion_matrix(
    personalized_y_test,
    personalized_model.predict(personalized_X_test),
    title='Personalized Model Confusion Matrix',
    output_path=str(PROJECT_ROOT / 'results' / 'figures' / 'personalized_confusion_matrix.png')
)